### Day 05 - Scikit Learn & Pipelines

### Setup & data loading

- We reuse the Titanic dataset from Day 4.
- Minimal cleaning here; most preprocessing will be handled by scikit-learn pipelines.
- We define:
  - Numeric feature columns
  - Categorical feature columns
  - Target: `survived`


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
notebook_path = Path.cwd()
repo_root = notebook_path
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

data_dir = repo_root / "data"
titanic_path = data_dir / "titanic.csv"

df = pd.read_csv(titanic_path)

# Rename to standard names if needed (adjust to your CSV)
df = df.rename(columns={
    "Survived": "survived",
    "Pclass": "pclass",
    "Age": "age",
    "SibSp": "sibsp",
    "Parch": "parch",
    "Fare": "fare",
    "Sex": "sex",
    "Embarked": "embarked"
})

# Create 'class' column
df["class"] = df["pclass"].map({1: "First", 2: "Second", 3: "Third"})

# Ensure numeric types
numeric_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop obvious non-feature columns if present
cols_to_drop = [c for c in ["deck", "embark_town", "alive", "who", "adult_male", "name", "ticket", "boat"] 
                if c in df.columns]
df = df.drop(columns=cols_to_drop)

# Basic cleaning
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.dropna(subset=["fare"])

# Define feature sets
numeric_features = ["age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked", "class"]

X = df[numeric_features + categorical_features]
y = df["survived"]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()


X shape: (891, 7)
y shape: (891,)


,age,sibsp,parch,fare,sex,embarked,class
0,22.0,1,0,7.2500,male,S,Third
1,38.0,1,0,71.2833,female,C,First
2,26.0,0,0,7.9250,female,S,Third
3,35.0,1,0,53.1000,female,S,First
4,35.0,0,0,8.0500,male,S,Third


### Why pipelines?

**Problems without pipelines:**

- Preprocessing code is repeated and easy to get wrong.
- Risk of **data leakage**: fitting scalers/encoders on the full dataset instead of only on training data.
- Hard to keep preprocessing consistent between train, validation, and test sets.
- Model + preprocessing are separate, making experimentation and deployment messy.

**What is a pipeline?**

- A `Pipeline` chains multiple steps:
  - Transformers (e.g. imputation, scaling, encoding)
  - Final estimator (model)
- When you call `fit` on the pipeline:
  - Each transformer is **fitted on the training data** and then applied.
  - The model is trained on the transformed data.
- When you call `predict`:
  - The same transformers are applied (using previously fitted parameters).
  - The model makes predictions.

**Benefits:**

- Prevents leakage by design.
- Ensures consistent preprocessing.
- Makes model selection and hyperparameter tuning cleaner.
- Easier to save/load a full “model + preprocessing” object for deployment.


### Simple numeric pipeline

This pipeline prepares numeric data and then trains a classification model.

It has three ordered steps:

1. **Imputer — `SimpleImputer(strategy="median")`**
   - Replaces missing numeric values (`NaN`) with the median value of that column.
   - The median is calculated from the training data when we call `fit`.

2. **Scaler — `StandardScaler()`**
   - Standardizes each numeric feature so it has approximately mean 0 and standard deviation 1.
   - Scaling is useful because features such as `age` and `fare` have very different ranges; logistic regression generally works better when numeric features are on comparable scales.

3. **Model — `LogisticRegression(max_iter=1000)`**
   - Learns to predict a binary target, such as Titanic survival (`0` = did not survive, `1` = survived).
   - It produces a probability of belonging to class 1, then converts that probability into a predicted class.
   - `max_iter=1000` gives the optimization algorithm enough iterations to converge.

`Pipeline` connects these steps into one object. When we call `fit(X_train, y_train)`, it first imputes missing values, then scales the data, then trains logistic regression. When we call `predict(X_test)`, it applies the *same training-derived* medians, means, and standard deviations before predicting.

This prevents data leakage because the test data is never used to calculate preprocessing statistics.


### Exercise - 1: numeric Titanic features

For this first example, we use only numeric Titanic columns:

- `age`
- `sibsp`
- `parch`
- `fare`

The target is `survived`.

The pipeline performs three jobs in order:

1. Fill missing numeric values with each column's median.
2. Scale each numeric feature to a similar range.
3. Train logistic regression to predict survival.

We pass raw data into the pipeline. We do not manually fill missing values or scale `X_train` and `X_test`.

When `pipeline.fit(X_train, y_train)` runs:
- The imputer learns medians from `X_train`.
- The scaler learns means and standard deviations from `X_train`.
- Logistic regression trains on the processed training data.

When `pipeline.predict(X_test)` runs:
- The pipeline uses the already learned training medians and scaling values.
- It transforms `X_test` consistently.
- It returns survival predictions.

This avoids leakage because information from the test set is not used while fitting preprocessing steps.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Select only numeric columns from the Titanic dataframe.
numeric_features = ["age", "sibsp", "parch", "fare"]

# X contains model inputs; y contains the value to predict.
X_numeric = df[numeric_features]
y = df["survived"]

# Split raw data before preprocessing.
# stratify=y keeps the survivor/non-survivor ratio similar in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X_numeric,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Build one object that preprocesses data and trains a classifier.
numeric_pipeline = Pipeline(steps=[
    # Replace missing values using medians learned from training data.
    ("imputer", SimpleImputer(strategy="median")),

    # Standardize numeric columns using training-set mean and standard deviation.
    ("scaler", StandardScaler()),

    # Learn to classify survival: 0 = no, 1 = yes.
    ("model", LogisticRegression(max_iter=1000))
])

# Fit every pipeline step using training data only.
numeric_pipeline.fit(X_train, y_train)

# Predict survival for unseen test rows.
y_pred = numeric_pipeline.predict(X_test)

# Compare predictions with the known test labels.
test_accuracy = accuracy_score(y_test, y_pred)

print("Test accuracy:", round(test_accuracy, 3))

# Print all named pipeline steps.
print(numeric_pipeline.named_steps)

# Access the trained logistic-regression model specifically.
logreg_model = numeric_pipeline.named_steps["model"]

# One coefficient for each numeric input feature.
coef_df = pd.DataFrame({
    "feature": numeric_features,
    "coefficient": logreg_model.coef_[0]
})

coef_df


Test accuracy: 0.665
{'imputer': SimpleImputer(strategy='median'), 'scaler': StandardScaler(), 'model': LogisticRegression(max_iter=1000)}


,feature,coefficient
0,age,-0.314814
1,sibsp,-0.306605
2,parch,0.097152
3,fare,0.963824


### Exercise - 2: Categorical-only pipeline

This exercise uses only categorical Titanic features:

- `sex`
- `embarked`
- `class`

Logistic regression needs numeric input, but these columns contain text categories. Therefore, the pipeline will:

1. Fill missing categories with the most common value.
2. One-hot encode each category into binary `0`/`1` columns.
3. Train logistic regression to predict `survived`.

For example, `sex` may become a column such as `sex_female`:

- Female → `1`
- Male → `0`

We split the raw data first. The pipeline learns the most common categories and available category values from training data only, then applies the same transformation to test data.


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Select only categorical columns.
categorical_features = ["sex", "embarked", "class"]

# X contains only categorical passenger information.
X_categorical = df[categorical_features]

# y is the target we want to predict.
y = df["survived"]

# Split raw categorical data into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X_categorical,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Build a pipeline for categorical data.
categorical_pipeline = Pipeline(steps=[
    # Fill missing text categories using the most frequent value in each column.
    ("imputer", SimpleImputer(strategy="most_frequent")),

    # Convert text categories into numeric 0/1 columns.
    # Unknown categories in test data are safely ignored.
    ("onehot", OneHotEncoder(handle_unknown="ignore")),

    # Train a binary classifier for survived (1) / not survived (0).
    ("model", LogisticRegression(max_iter=1000))
])

# Learn preprocessing rules and train the model using training data only.
categorical_pipeline.fit(X_train, y_train)

# Predict survival for previously unseen test passengers.
y_pred = categorical_pipeline.predict(X_test)

# Calculate how many test predictions were correct.
test_accuracy = accuracy_score(y_test, y_pred)

print("Test accuracy:", round(test_accuracy, 3))


Test accuracy: 0.765


In [14]:
"""
Exercise: Combined numeric + categorical Titanic pipeline

This cell:
- Selects numeric and categorical columns from the Titanic dataframe.
- Splits data into train and test sets.
- Builds a ColumnTransformer that:
    - Imputes and scales numeric features.
    - Imputes and one-hot encodes categorical features.
- Wraps the preprocessor and a logistic regression model in a Pipeline.
- Fits the pipeline on training data.
- Evaluates accuracy on the test set.
"""

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -------------------------
# 1. Select feature columns
# -------------------------

# Numeric features to use in the model.
numeric_features = ["age", "sibsp", "parch", "fare"]

# Categorical features to use in the model.
categorical_features = ["sex", "embarked", "class"]

# Combine them into a single list for X.
all_features = numeric_features + categorical_features

# X contains all model inputs; y is the target we want to predict.
X = df[all_features]
y = df["survived"]

# --------------------------------
# 2. Train/test split (raw data)
# --------------------------------

# Split raw data before any preprocessing.
# stratify=y keeps the survivor ratio similar in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -------------------------------------------
# 3. Define numeric preprocessing sub-pipeline
# -------------------------------------------

# Numeric transformer:
# - Fill missing numeric values with the median of each column.
# - Scale features to mean 0, std 1.
numeric_transformer = Pipeline(steps=[
    ("num_imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# ----------------------------------------------
# 4. Define categorical preprocessing sub-pipeline
# ----------------------------------------------

# Categorical transformer:
# - Fill missing categories with the most frequent value.
# - One-hot encode categories into 0/1 columns.
categorical_transformer = Pipeline(steps=[
    ("cat_imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# -------------------------------------------------
# 5. Combine numeric and categorical preprocessing
# -------------------------------------------------

# ColumnTransformer applies different transformers to different column groups.
preprocessor = ColumnTransformer(
    transformers=[
        # Apply numeric_transformer to numeric_features columns.
        ("num", numeric_transformer, numeric_features),
        # Apply categorical_transformer to categorical_features columns.
        ("cat", categorical_transformer, categorical_features)
    ],
    # Drop any columns not explicitly listed above.
    remainder="drop"
)

# -------------------------------------------------
# 6. Wrap preprocessor + model in a Pipeline
# -------------------------------------------------

# Full pipeline:
# 1. Preprocess data (imputation + scaling + encoding).
# 2. Train logistic regression on the processed features.
clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# -------------------------------------------------
# 7. Fit the pipeline on training data
# -------------------------------------------------

# This single call:
# - Fits the preprocessor on X_train.
# - Transforms X_train.
# - Trains logistic regression on the transformed data.
clf.fit(X_train, y_train)

# -------------------------------------------------
# 8. Evaluate on the test set
# -------------------------------------------------

# Predict survival labels for test passengers.
y_pred = clf.predict(X_test)

# Predict probability of survival (class 1) for each test passenger.
y_proba = clf.predict_proba(X_test)[:, 1]

# Compute evaluation metrics.
test_accuracy = accuracy_score(y_test, y_pred)

print("Test accuracy:", round(test_accuracy, 3))

print("\nClassification report:\n")
print(classification_report(y_test, y_pred, digits=3))

print("Confusion matrix:\n")
print(confusion_matrix(y_test, y_pred))


Test accuracy: 0.804

Classification report:

              precision    recall  f1-score   support

           0      0.810     0.891     0.848       110
           1      0.793     0.667     0.724        69

    accuracy                          0.804       179
   macro avg      0.802     0.779     0.786       179
weighted avg      0.803     0.804     0.801       179

Confusion matrix:

[[98 12]
 [23 46]]


### Cross-validation

**Why cross-validation?**

- A single train/test split can be noisy (depends on the random split).
- Cross-validation (CV) gives a more stable estimate of generalization performance.

**k-fold CV:**

- Split data into *k* folds.
- For each fold:
  - Train on *k-1* folds.
  - Evaluate on the held-out fold.
- Average metrics across folds.

**Stratified k-fold:**

- Preserves class proportions in each fold.
- Recommended for classification with imbalanced targets.

**What we’ll do:**

- Use `StratifiedKFold` with `k=5`.
- Evaluate:
  - Logistic Regression pipeline
  - Random Forest pipeline
- Metrics: accuracy, ROC AUC, F1.


### Cross-validation with the full pipeline

A single train/test split gives one performance result, which can change depending on the random split.

Cross-validation evaluates the model multiple times:

1. Split the dataset into 5 folds.
2. Train the full pipeline on 4 folds.
3. Evaluate it on the remaining fold.
4. Repeat until every fold has been the validation fold once.
5. Calculate the mean and standard deviation of each metric.

We pass the complete pipeline (`clf`) to cross-validation, not already transformed data.

For every fold, scikit-learn automatically:
- Fits the imputer, scaler, and encoder using that fold's training portion only.
- Trains logistic regression.
- Evaluates on that fold's validation portion.

This keeps preprocessing leakage-safe.


In [15]:
"""
Cross-validation for the complete Titanic pipeline.

This cell reuses:
- clf: the Pipeline containing preprocessor + LogisticRegression
- X: the raw DataFrame with numeric and categorical Titanic features
- y: the survived target column
"""

from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd

# Create 5 stratified folds.
# Stratified means each fold keeps a similar survived/non-survived ratio.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Metrics to calculate for every validation fold.
scoring = [
    "accuracy",  # Fraction of predictions that are correct.
    "roc_auc",   # How well predicted probabilities rank survivors above non-survivors.
    "f1"         # Balance between precision and recall for survived = 1.
]

# Run cross-validation.
# Do NOT call clf.fit() before this for CV purposes:
# cross_validate fits a fresh copy of the pipeline separately for each fold.
cv_results = cross_validate(
    estimator=clf,
    X=X,
    y=y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

# Create a readable table: one row for each fold.
cv_scores_df = pd.DataFrame({
    "fold": range(1, 6),
    "accuracy": cv_results["test_accuracy"],
    "roc_auc": cv_results["test_roc_auc"],
    "f1": cv_results["test_f1"]
})

print("Scores for each fold:")
display(cv_scores_df)

# Calculate mean and standard deviation across the five folds.
summary_df = pd.DataFrame({
    "metric": ["accuracy", "roc_auc", "f1"],
    "mean": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_roc_auc"].mean(),
        cv_results["test_f1"].mean()
    ],
    "std": [
        cv_results["test_accuracy"].std(),
        cv_results["test_roc_auc"].std(),
        cv_results["test_f1"].std()
    ]
})

print("\nCross-validation summary:")
display(summary_df.round(3))


Scores for each fold:


,fold,accuracy,roc_auc,f1
0,1,0.782123,0.874111,0.715328
1,2,0.803371,0.850401,0.728682
2,3,0.797753,0.826671,0.704918
3,4,0.780899,0.826671,0.715328
4,5,0.820225,0.879404,0.764706



Cross-validation summary:


,metric,mean,std
0,accuracy,0.797,0.015
1,roc_auc,0.851,0.022
2,f1,0.726,0.021


### Hyperparameter tuning with GridSearchCV

So far, we used:

- A fixed model: `LogisticRegression(max_iter=1000)`
- A fixed preprocessor: median imputation + scaling + one-hot encoding

These choices involve **hyperparameters**: settings you choose before training, such as:

- Regularization strength in logistic regression (`C`)
- Type of regularization (`penalty`)
- Number of trees, tree depth, etc. (for tree-based models)

**GridSearchCV** tries many combinations of hyperparameters:

1. You define a **parameter grid**: which hyperparameters to try and which values.
2. For each combination:
   - It runs cross-validation (e.g. 5 folds).
   - Computes the chosen metric (e.g. ROC AUC).
3. It picks the combination with the best average CV score.

Key idea:

- The pipeline (preprocessing + model) is treated as one object.
- Hyperparameters are specified with names like:
  - `"model__C"` (C for the model step)
  - `"model__penalty"`
- GridSearchCV handles refitting the preprocessor and model for each fold and each parameter combination.

This gives you a systematic way to improve performance instead of guessing hyperparameters.


### Hyperparameter tuning with GridSearchCV

A model learns **parameters** from data.

For logistic regression, learned parameters include the coefficient for each feature.

A **hyperparameter** is chosen before training. It controls how the model learns.

In logistic regression, `C` controls regularization strength:

- Smaller `C` → stronger regularization, simpler model.
- Larger `C` → weaker regularization, model can fit the training data more closely.

`GridSearchCV` automatically:

1. Tries every value in a chosen parameter grid.
2. Uses cross-validation to evaluate every choice.
3. Selects the setting with the best average score.
4. Refits the best pipeline on the complete training data.

Because our estimator is a pipeline, we write:

`model__C`

- `model` is the name of the Logistic Regression pipeline step.
- `C` is the Logistic Regression hyperparameter.
- The double underscore `__` means “parameter inside this pipeline step.”

Important:
- Run grid search only on `X_train` and `y_train`.
- Keep `X_test` and `y_test` untouched until final evaluation.


In [16]:
"""
Grid search for Logistic Regression.

This cell reuses:
- clf: complete pipeline with "preprocess" and "model" steps
- X_train, X_test, y_train, y_test: full mixed-feature train/test split

GridSearchCV will test several values of LogisticRegression's C parameter
using cross-validation on the training data only.
"""

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score

# Create 5 stratified validation folds from training data.
# Each fold keeps a similar survival ratio.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Values of C that GridSearchCV will test.
# Total combinations here = 4 because there is one hyperparameter.
param_grid = {
    "model__C": [0.01, 0.1, 1, 10]
}

# Create the search object.
grid_search = GridSearchCV(
    estimator=clf,          # Reuse full preprocessing + Logistic Regression pipeline.
    param_grid=param_grid,  # Values to try.
    scoring="roc_auc",      # Choose the best C by average ROC AUC.
    cv=cv,                  # Use the 5 folds defined above.
    n_jobs=-1,              # Use available CPU cores.
    return_train_score=True
)

# Run grid search on TRAINING data only.
# Each fold independently fits the entire pipeline, including preprocessing.
grid_search.fit(X_train, y_train)

# Print the best average validation ROC AUC.
print("Best CV ROC AUC:", round(grid_search.best_score_, 3))

# Print the best C value.
print("Best parameters:", grid_search.best_params_)


Best CV ROC AUC: 0.853
Best parameters: {'model__C': 1}


In [17]:
"""
Final test evaluation.

The test set was not used while selecting C.
Now we evaluate the best pipeline exactly once on X_test.
"""

# Get the fitted pipeline with the best C value.
best_clf = grid_search.best_estimator_

# Predict classes and survival probabilities on the held-out test set.
y_pred_best = best_clf.predict(X_test)
y_proba_best = best_clf.predict_proba(X_test)[:, 1]

# Evaluate final performance.
test_accuracy = accuracy_score(y_test, y_pred_best)
test_roc_auc = roc_auc_score(y_test, y_proba_best)

print("Test accuracy:", round(test_accuracy, 3))
print("Test ROC AUC:", round(test_roc_auc, 3))


Test accuracy: 0.804
Test ROC AUC: 0.843


### How to choose the range of values for `C` in the parameter grid

#### What `C` means

In scikit-learn’s `LogisticRegression`:

- `C` is the **inverse of regularization strength**.
- **Smaller `C`** → stronger regularization → simpler model, coefficients closer to zero.
- **Larger `C`** → weaker regularization → more flexible model, can fit the training data more closely.
- Default value: `C = 1.0`.
- `C = np.inf` means **no regularization**.

You can use LogisticRegressionCV as shortcut as it automatically searches over a range of C values using crossvalidation.


In [18]:

from sklearn.linear_model import LogisticRegressionCV

# LogisticRegressionCV:
# - Automatically searches over multiple C values using cross-validation.
# - Cs=10 means: try 10 values on a log scale (default range is roughly 1e-4 to 1e4).
# - cv=5 means: use 5-fold cross-validation on the training data.
# - scoring="roc_auc" means: choose the best C based on ROC AUC.
# - max_iter=1000 allows enough iterations for convergence.
log_reg_cv = LogisticRegressionCV(
    Cs=10,
    cv=5,
    scoring="roc_auc",
    max_iter=1000,
    n_jobs=-1
)

# -------------------------------------------------
# 7. Wrap preprocessor + model in a Pipeline
# -------------------------------------------------

# Full pipeline:
# 1. Preprocess data (imputation + scaling + encoding).
# 2. Run LogisticRegressionCV, which internally performs CV over C.
clf_cv = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", log_reg_cv)
])

# -------------------------------------------------
# 8. Fit the pipeline on training data
# -------------------------------------------------

# This single call:
# - Fits the preprocessor on X_train.
# - Transforms X_train.
# - Runs internal cross-validation over C values.
# - Refits Logistic Regression with the best C on the full training set.
clf_cv.fit(X_train, y_train)

# -------------------------------------------------
# 9. Evaluate on the test set
# -------------------------------------------------

# Predict survival labels for test passengers.
y_pred = clf_cv.predict(X_test)

# Predict probability of survival (class 1) for each test passenger.
y_proba = clf_cv.predict_proba(X_test)[:, 1]

# Compute evaluation metrics.
test_accuracy = accuracy_score(y_test, y_pred)
test_roc_auc = roc_auc_score(y_test, y_proba)

print("Test accuracy:", round(test_accuracy, 3))
print("Test ROC AUC:", round(test_roc_auc, 3))

# -------------------------------------------------
# 10. Inspect the chosen C value
# -------------------------------------------------

# Access the fitted LogisticRegressionCV step.
log_reg_cv_fitted = clf_cv.named_steps["model"]

# Best C value selected by cross-validation.
best_C = log_reg_cv_fitted.C_[0]  # C_ is an array; for binary classification we take the first element.

print("\nBest C chosen by LogisticRegressionCV:", best_C)

# Optional: see all C values that were tested.
all_C_values = log_reg_cv_fitted.Cs_
print("C values tested:", np.round(all_C_values, 4))


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details

Test accuracy: 0.804
Test ROC AUC: 0.844

Best C chosen by LogisticRegressionCV: 0.046415888336127774
C values tested: [1.0000000e-04 8.0000000e-04 6.0000000e-03 4.6400000e-02 3.5940000e-01
 2.7826000e+00 2.1544300e+01 1.6681010e+02 1.2915497e+03 1.0000000e+04]
